<h1 style="color:#2E86AB; font-family:Georgia; text-align:center; border-bottom: 3px solid #2E86AB; padding-bottom:10px;">
🤖 Modelo Predictivo de Deserción Escolar
</h1>

**Autor:** Eddy Luis  
**Fecha:** Mayo 2026  
**Herramientas:** Python · Scikit-learn · XGBoost · SHAP

---

## 📌 Descripción
Entrenamos un modelo de clasificación que predice el **nivel de riesgo de deserción** (muy_bajo, bajo, medio, alto) para cada centro educativo, usando características del centro y contexto de su distrito.

---

## 📋 Contenido de este Notebook

1. 📦 Importar Librerías y Cargar Datos
2. 🎯 Preparar Features y Target
3. ✂️ Train / Test Split
4. 🌲 Modelo 1: Random Forest
5. ⚡ Modelo 2: XGBoost
6. 🏆 Comparación de Modelos
7. 🔍 Interpretabilidad con SHAP
8. 💾 Guardar Modelo Final

---

<h2 style="color:#A23B72; font-family:Georgia;">
📦 1. Importar Librerías y Cargar Datos
</h2>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import shap
import joblib
import warnings
warnings.filterwarnings("ignore")

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "dataset_modelo.csv")
print(f"Dataset cargado: {df.shape}")
print(f"\nDistribución de riesgo:\n{df['riesgo'].value_counts()}")

<h2 style="color:#A23B72; font-family:Georgia;">
🎯 2. Preparar Features y Target
</h2>

**Target:** `riesgo` (muy_bajo / bajo / medio / alto)

**Features usadas:** características del centro + contexto del distrito + tendencia histórica. Se excluyen las tasas de abandono del período actual para evitar *data leakage* (ya que el target se derivó de ellas).

Se eliminan los 2,306 centros de educación de adultos (`sin_datos`) que no tienen tasa de abandono comparable.

In [ ]:
# Filtrar centros sin datos (educación de adultos)
df_model = df[df["riesgo"] != "sin_datos"].copy()
print(f"Centros para modelado: {len(df_model)} (eliminados {len(df) - len(df_model)} de adultos)")

FEATURES = [
    "matricula",
    "es_publico",
    "tiene_inicial",
    "tiene_primario",
    "tiene_secundario",
    "num_niveles",
    "coordenadas longitud",
    "num_centros",
    "matricula_total",
    "matricula_promedio",
    "matricula_mediana",
    "pct_publico",
    "tendencia_abandono_sec",
    "abandono_sec_historico",
]

X = df_model[FEATURES].copy()
y = df_model["riesgo"].copy()

# Imputar nulos en coordenadas longitud (325 nulos) con la mediana
X["coordenadas longitud"] = X["coordenadas longitud"].fillna(X["coordenadas longitud"].median())

# Codificar target
le = LabelEncoder()
le.fit(["muy_bajo", "bajo", "medio", "alto"])
y_encoded = le.transform(y)

print(f"\nFeatures: {len(FEATURES)}")
print(f"Clases: {le.classes_}")
print(f"\nDistribución:\n{pd.Series(y_encoded).value_counts().sort_index()}")
print(f"  0={le.classes_[0]}, 1={le.classes_[1]}, 2={le.classes_[2]}, 3={le.classes_[3]}")
print(f"\nNulos en X: {X.isnull().sum().sum()}")

<h2 style="color:#A23B72; font-family:Georgia;">
✂️ 3. Train / Test Split
</h2>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")
print(f"\nDistribución en train:")
for i, cls in enumerate(le.classes_):
    n = (y_train == i).sum()
    print(f"  {cls}: {n} ({n/len(y_train)*100:.1f}%)")

<h2 style="color:#A23B72; font-family:Georgia;">
🌲 4. Modelo 1: Random Forest
</h2>

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("=== Random Forest ===")
print(f"Accuracy: {rf.score(X_test, y_test):.4f}")
print(f"\nCross-validation (5-fold): {cross_val_score(rf, X, y_encoded, cv=5, scoring='accuracy').mean():.4f}")
print(f"\n{classification_report(y_test, y_pred_rf, target_names=le.classes_)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, display_labels=le.classes_, ax=ax, cmap="Blues"
)
ax.set_title("Random Forest — Matriz de Confusión", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

<h2 style="color:#A23B72; font-family:Georgia;">
⚡ 5. Modelo 2: XGBoost
</h2>

In [ ]:
# Calcular pesos para clases desbalanceadas
from collections import Counter
counts = Counter(y_train)
total = len(y_train)
sample_weights = np.array([total / (len(counts) * counts[c]) for c in y_train])

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    verbosity=0,
)
xgb.fit(X_train, y_train, sample_weight=sample_weights)

y_pred_xgb = xgb.predict(X_test)
print("=== XGBoost ===")
print(f"Accuracy: {xgb.score(X_test, y_test):.4f}")
print(f"\nCross-validation (5-fold): {cross_val_score(xgb, X, y_encoded, cv=5, scoring='accuracy').mean():.4f}")
print(f"\n{classification_report(y_test, y_pred_xgb, target_names=le.classes_)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb, display_labels=le.classes_, ax=ax, cmap="Oranges"
)
ax.set_title("XGBoost — Matriz de Confusión", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()